# Labo 36 · Carnet Colab — Un transformeur Keras

Classer les critiques IMDB avec un petit encodeur de transformeur, et le comparer au LSTM du labo 31.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
L, V = 200, 10000
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=V)
X_train = keras.utils.pad_sequences(x_train, maxlen=L)
X_test = keras.utils.pad_sequences(x_test, maxlen=L)
index = keras.datasets.imdb.get_word_index()
mots = {i + 3: m for m, i in index.items()}
print(X_train.shape)

## 1. Les briques

Deux couches sur mesure : `PlongementPosition` additionne le plongement de chaque mot et un plongement **appris** de sa position; `BlocTransformeur` fait l'attention multi-têtes et le réseau dense, chacun avec sa connexion résiduelle et sa normalisation par couche.

**À toi** : complète `call` du bloc, en suivant le cours (ici, la normalisation est appliquée **après** l'addition, comme dans le transformeur original).

In [ ]:
class PlongementPosition(layers.Layer):
    def __init__(self, longueur, vocabulaire, d, **kw):
        super().__init__(**kw)
        self.mots = layers.Embedding(vocabulaire, d)
        self.positions = layers.Embedding(longueur, d)
    def call(self, x):
        positions = keras.ops.arange(0, keras.ops.shape(x)[-1])
        return self.mots(x) + self.positions(positions)

class BlocTransformeur(layers.Layer):
    def __init__(self, d, tetes, d_ff, **kw):
        super().__init__(**kw)
        self.attention = layers.MultiHeadAttention(num_heads=tetes, key_dim=d // tetes)
        self.dense = keras.Sequential([layers.Dense(d_ff, activation="relu"), layers.Dense(d)])
        self.norme1, self.norme2 = layers.LayerNormalization(), layers.LayerNormalization()
        self.abandon1, self.abandon2 = layers.Dropout(0.1), layers.Dropout(0.1)
    def call(self, x, training=False):
        pass   # À toi : attention (x, x), abandon, x + ..., norme1; puis dense, abandon, + ..., norme2

## 2. Le classificateur

**À toi** : assemble le modèle avec l'API fonctionnelle (cours 28) : entrée de longueur L, `PlongementPosition(L, V, 32)`, `BlocTransformeur(32, 2, 64)`, une moyenne sur les positions (`layers.GlobalAveragePooling1D()`), un dropout de 0.1, `Dense(20, relu)`, puis la sortie binaire. Entraîne 3 époques (lots de 64, `validation_split=0.2`) en mesurant le temps, puis évalue.

In [ ]:
# À toi
modele = None

## 3. Contre le LSTM

**À toi** : le modèle `Embedding(V, 32)`, `LSTM(64)`, `Dense(1, sigmoid)` du labo 31, entraîné de la même façon. Compare paramètres, temps et exactitude.

In [ ]:
# À toi

## 4. Où regarde l'attention?

`MultiHeadAttention` peut retourner ses poids. **À toi** : pour la première critique du test, calcule les poids d'attention du bloc entraîné (moyenne sur les têtes et sur les positions qui regardent), et affiche les 10 mots qui reçoivent le plus d'attention, en ignorant le remplissage (les 0).

In [ ]:
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : le nombre de paramètres, l'exactitude au test et le temps par époque du transformeur et du LSTM, et les trois premiers mots qui reçoivent le plus d'attention dans la critique.